# 09 Batch Size

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp_cmb008_s6_batch_size.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s6_batch_size.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb008_s6_batch_aggregate.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s6_batch_aggregate.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Combined/scripts/exp_cmb008_s6_batch_size.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '81eda4abbd0da5ffe139175b748e8cdb5a12234f'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb008_s6_batch_aggregate.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '9e6487d9b26463d03a746d32aca6454922bb9ef4'}}

## exp_cmb008_s6_batch_size

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s6_batch_size.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s6_batch_size.py)  
Git blob: `81eda4abbd0da5ffe139175b748e8cdb5a12234f`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

CMB008 S6: change Adam minibatch size ONLY (200, 128, 64).

One independent model per (seed 42/43/44, leave-one-writer-out 41/42/43/44,
batch_size 200/128/64). Entire 78,720 training candidate instances used.
early_stopping=False, max_iter=200, n_iter_no_change=10 (sklearn default).
No independent test, Professor Test, or new 480 used.

CLI 인자 (정적 추출): `--batch-size`, `--seed`, `--heldout`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s6_batch_size.py
#!/usr/bin/env python3
"""CMB008 S6: change Adam minibatch size ONLY (200, 128, 64).

One independent model per (seed 42/43/44, leave-one-writer-out 41/42/43/44,
batch_size 200/128/64). Entire 78,720 training candidate instances used.
early_stopping=False, max_iter=200, n_iter_no_change=10 (sklearn default).
No independent test, Professor Test, or new 480 used.
"""
from __future__ import annotations
import argparse,csv,json,time,warnings
from pathlib import Path
import numpy as np
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from exp_cmb008_s5_early_stopping import (
    ROOT,WRITERS,LABELS,V3_SHA,HANDMADE_SHA,
    load_official,load_writers,PixelScaler255,prepareraw,
    process,fixed_scale
)

OFFICIAL=ROOT/"02_data"/"datasets"/"MNIST_EMNIST_official_split_112000.npz"
HANDMADE=ROOT/"02_data"/"Hand_made"
WATCH=((13,0,"D_to_0"),(11,8,"B_to_8"),(8,11,"8_to_B"),(0,13,"0_to_D"))
FOCUS=(0,8,11,13)

def savecsv(path,rows):
    assert rows
    with path.open("w",encoding="utf-8-sig",newline="") as f:
        wr=csv.DictWriter(f,fieldnames=list(rows[0]))
        wr.writeheader();wr.writerows(rows)

def main():
    p=argparse.ArgumentParser()
    p.add_argument("--batch-size",choices=(200,128,64),type=int,required=True)
    p.add_argument("--seed",choices=(42,43,44),type=int,required=True)
    p.add_argument("--heldout",choices=WRITERS,type=int,required=True)
    p.add_argument("--out",type=Path,required=True)
    a=p.parse_args()
    assert not a.out.exists(),a.out
    a.out.mkdir(parents=True)
    official=load_official(OFFICIAL)
    writers=load_writers(HANDMADE)
    assert official["sha256"]==V3_SHA
    for w in WRITERS:assert writers[w]["sha256"]==HANDMADE_SHA[w]
    assert len(official["X_train"])==76800 and len(official["X_val"])==19200
    start=time.perf_counter()
    ox,wx,aug,_=prepareraw(official,writers,"scale30_stroke28")
    remain=[w for w in WRITERS if w!=a.heldout]
    X=np.concatenate([ox]+[wx[w] for w in remain])
    y=np.concatenate([official["y_train"]]+[writers[w]["y"] for w in remain])
    assert X.shape==(78720,784) and y.shape==(78720,)
    val_x=process(official["X_val"],"com",20)
    val_y=np.asarray(official["y_val"],dtype=int)
    held_x=wx[a.heldout]
    held_y=np.asarray(writers[a.heldout]["y"],dtype=int)
    assert len(held_y)==640 and held_y.shape==(640,)
    net=MLPClassifier(random_state=a.seed,hidden_layer_sizes=(256,),
        alpha=.0001,learning_rate_init=.001,solver="adam",activation="relu",
        early_stopping=False,batch_size=a.batch_size,max_iter=200,
        n_iter_no_change=10,tol=.0001)
    model=Pipeline([("scale_0_1",PixelScaler255()),("model",net)])
    with warnings.catch_warnings(record=True) as got:
        warnings.simplefilter("always")
        tick=time.perf_counter()
        model.fit(X,y)
        elapsed=time.perf_counter()-tick
    assert net.n_features_in_==784
    assert np.array_equal(net.classes_,np.arange(16))
    yp=np.asarray(model.predict(held_x),dtype=int)
    vp=np.asarray(model.predict(val_x),dtype=int)
    assert len(yp)==640 and len(vp)==19200
    predictions=[{"batch_size":a.batch_size,"seed":a.seed,
      "writer":a.heldout,"index":i,"actual":LABELS[int(held_y[i])],
      "pred":LABELS[int(yp[i])]} for i in range(640)]
    savecsv(a.out/"oof_predictions.csv",predictions)
    focus={}
    for class_id in range(16):
        mask=held_y==class_id
        focus[LABELS[class_id]]={
            "n":int(mask.sum()),
            "errors":int(np.count_nonzero(yp[mask]!=class_id))}
    assert all(v["n"]==40 for v in focus.values())
    pairs={key:int(np.count_nonzero((held_y==aa)&(yp==bb)))
        for aa,bb,key in WATCH}
    # Calculate original scale-sensitivity gate: worst class-specific
    # family-flip on OOF/OfficialVal under 0.8 and 1.2 scale, using CoM.
    # Predict_proba classes 0-9 vs A-F; family flip threshold 0.5.
    sensitivity=[]
    for ds,raw_data,yy,base in (
        ("OOF",writers[a.heldout]["X"],held_y,held_x),
        ("OfficialVal",official["X_val"],val_y,val_x)):
        original_family=(model.predict_proba(base)[:,10:].sum(axis=1)>=.5)
        for scale in (.8,1.2):
            xx=process(fixed_scale(raw_data,scale),"com",20)
            scaled_family=(model.predict_proba(xx)[:,10:].sum(axis=1)>=.5)
            for cl in range(16):
                mask=yy==cl
                sensitivity.append({"dataset":ds,"writer":a.heldout,
                  "seed":a.seed,"scale":scale,"actual":LABELS[cl],
                  "n":int(mask.sum()),
                  "flip_count":int(np.count_nonzero((original_family!=scaled_family)&mask))})
    savecsv(a.out/"class_family_flip.csv",sensitivity)
    meta={"experiment":"EXP-CMB-008","stage":"S6_batch_size",
        "batch_size":a.batch_size,"seed":a.seed,"heldout":a.heldout,
        "frozen":{"hidden_layer_sizes":[256],"alpha":.0001,
          "learning_rate_init":.001,"solver":"adam","activation":"relu",
          "early_stopping":False,"max_iter":200,
          "tol":.0001,"n_iter_no_change":10},
        "n_training":len(y),"n_heldout":len(held_y),
        "n_official_val":len(val_y),
        "official_sha256":official["sha256"],
        "handmade_sha256":{str(w):writers[w]["sha256"] for w in WRITERS},
        "augmentation":aug,"augmentation_seed":20261116,
        "handmade_train_preprocess":"CoM-only",
        "heldout_and_official_val_preprocess":"CoM-only",
        "writer_heldout_not_used_to_train":True,
        "independent_test_used":False,"professor_test_used":False,
        "new_handmade_480_used":False,
        "writer_oof_accuracy":float(np.mean(yp==held_y)),
        "official_val_accuracy":float(np.mean(vp==val_y)),
        "focus_class":focus,"directional_pairs":pairs,
        "n_iter":int(net.n_iter_),"first_loss":float(net.loss_curve_[0]),
        "last_loss":float(net.loss_curve_[-1]),
        "fit_seconds":elapsed,
        "wall_seconds":time.perf_counter()-start,
        "sklearn_version":sklearn.__version__,
        "warnings":[str(w.message) for w in got]}
    (a.out/"manifest.json").write_text(json.dumps(meta,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("BATCH_FOLD_OK",json.dumps({"batch":a.batch_size,"seed":a.seed,
      "heldout":a.heldout,"oof":meta["writer_oof_accuracy"],
      "official":meta["official_val_accuracy"],"pairs":pairs,
      "n_iter":net.n_iter_,"seconds":round(elapsed,1)},ensure_ascii=False),flush=True)
if __name__=="__main__":
    main()


## exp_cmb008_s6_batch_aggregate

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb008_s6_batch_aggregate.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb008_s6_batch_aggregate.py)  
Git blob: `9e6487d9b26463d03a746d32aca6454922bb9ef4`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

S6: audit all 36 paired batch-size fits, evaluate frozen CMB008 gates.

Final deployed/baseline HP are NEVER automatically changed. Outputs a
research-only recommended S6 candidate; user must approve any HP adoption.

CLI 인자 (정적 추출): `--incoming`, `--s5-original`, `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb008_s6_batch_aggregate.py
#!/usr/bin/env python3
"""S6: audit all 36 paired batch-size fits, evaluate frozen CMB008 gates.

Final deployed/baseline HP are NEVER automatically changed. Outputs a
research-only recommended S6 candidate; user must approve any HP adoption.
"""
from __future__ import annotations
import argparse,csv,json,statistics
from collections import Counter,defaultdict
from pathlib import Path

BATCHES=(200,128,64)
SEEDS=(42,43,44)
WRITERS=(41,42,43,44)
PAIRS=("D_to_0","B_to_8","8_to_B","0_to_D")
FOCUS=("0","8","B","D")
def csvread(p):
    with p.open(encoding="utf-8-sig",newline="") as f:return list(csv.DictReader(f))
def csvwrite(p,rows):
    assert rows
    with p.open("w",encoding="utf-8-sig",newline="") as f:
        wr=csv.DictWriter(f,fieldnames=list(rows[0]))
        wr.writeheader();wr.writerows(rows)
def acc_str(x):return f"{100*x:.3f}%"
def score_for(batch,artifacts):
    records=[(s,w,artifacts[(batch,s,w)]) for s in SEEDS for w in WRITERS]
    pred=[(s,w,row) for s,w,d in records for row in d["pred"]]
    assert len(pred)==7680
    n_correct=sum(r["actual"]==r["pred"] for _,_,r in pred)
    assert all(Counter(r["actual"] for _,_,r in pred)[c]==480 for c in "0123456789ABCDEF")
    oof=n_correct/7680
    accuracy_seed=[sum(r["actual"]==r["pred"] for z,w,r in pred if z==s)/2560 for s in SEEDS]
    writers={str(w):sum(r["actual"]==r["pred"] for z,z_w,r in pred if z_w==w)/1920 for w in WRITERS}
    val=statistics.mean(float(d["m"]["official_val_accuracy"]) for _,_,d in records)
    pairs={key:sum(r["actual"]==actual and r["pred"]==predicted for _,_,r in pred)
      for key,actual,predicted in
      (("D_to_0","D","0"),("B_to_8","B","8"),("8_to_B","8","B"),("0_to_D","0","D"))}
    classes={cl:sum(r["actual"]==cl and r["pred"]!=cl for _,_,r in pred)/480 for cl in "0123456789ABCDEF"}
    family=defaultdict(lambda:[0,0])
    for _,_,d in records:
        for row in d["flip"]:
            k=(row["dataset"],row["actual"],float(row["scale"]))
            family[k][0]+=int(row["flip_count"])
            family[k][1]+=int(row["n"])
    assert len(family)==2*16*2
    flips={ds:max(count/total for (d,cl,sc),(count,total) in family.items() if d==ds)
           for ds in ("OOF","OfficialVal")}
    return {"batch_size":batch,"oof":oof,"seed_std":statistics.pstdev(accuracy_seed),
       "official_val":val,"writers":writers,"pairs":pairs,
       "classerror":classes,"flip":flips,
       "mean_epochs":statistics.mean(int(d["m"]["n_iter"]) for _,_,d in records),
       "mean_fit_seconds":statistics.mean(float(d["m"]["fit_seconds"]) for _,_,d in records),
       "warnings":sum(bool(d["m"]["warnings"]) for _,_,d in records)}
def gates(base,candidate):
    return {
      "oof_at_least_baseline":candidate["oof"]>=base["oof"]-1e-12,
      "official_val_drop_at_most_0p5pp":candidate["official_val"]>=base["official_val"]-.005-1e-12,
      "writer_drop_at_most_1pp":all(candidate["writers"][w]>=base["writers"][w]-.01-1e-12 for w in base["writers"]),
      "class_0_8_B_D_error_nonincrease":all(candidate["classerror"][cl]<=base["classerror"][cl]+1e-12 for cl in FOCUS),
      "worst_class_family_flip_at_most_plus_5pp":all(candidate["flip"][ds]<=base["flip"][ds]+.05+1e-12 for ds in ("OOF","OfficialVal")),
      "D_to_0_nonincrease":candidate["pairs"]["D_to_0"]<=base["pairs"]["D_to_0"],
      "B_to_8_nonincrease":candidate["pairs"]["B_to_8"]<=base["pairs"]["B_to_8"],
      "8_to_B_nonincrease":candidate["pairs"]["8_to_B"]<=base["pairs"]["8_to_B"],
      "0_to_D_nonincrease":candidate["pairs"]["0_to_D"]<=base["pairs"]["0_to_D"]}
def target(m):return all(m["pairs"][key]<=23 for key in PAIRS)
def compare_predictions(ref,other):
    fixed=induced=0
    for seed in SEEDS:
        for writer in WRITERS:
            aa=ref[(seed,writer)]["pred"]
            bb=other[(seed,writer)]["pred"]
            assert len(aa)==len(bb)==640
            for i in range(640):
                assert aa[i]["actual"]==bb[i]["actual"]
                assert int(aa[i]["index"])==int(bb[i]["index"])==i
                old=aa[i]["actual"]==aa[i]["pred"]
                new=bb[i]["actual"]==bb[i]["pred"]
                fixed+=int(not old and new)
                induced+=int(old and not new)
    return {"fixed":fixed,"induced":induced,"net_gain":fixed-induced}
def main():
    p=argparse.ArgumentParser()
    p.add_argument("--incoming",type=Path,required=True)
    p.add_argument("--s5-original",type=Path,required=True)
    p.add_argument("--out",type=Path,required=True)
    a=p.parse_args()
    artifacts={}
    original=json.loads(a.s5_original.read_text(encoding="utf-8"))
    assert original["stage"]=="S5_early_stopping"
    assert original["selected_early_stopping"] is False
    orig=original["scores"]["False"]
    observed_dirs={k:orig["pairs"][k.replace("_to_","->")] for k in PAIRS}
    assert orig["oof"]==0.9221354166666667
    for batch in BATCHES:
        for seed in SEEDS:
            for writer in WRITERS:
                d=a.incoming/f"cmb008-s6-bs{batch}-s{seed}-w{writer}"
                assert d.is_dir(),d
                m=json.loads((d/"manifest.json").read_text(encoding="utf-8"))
                assert m["stage"]=="S6_batch_size" and m["batch_size"]==batch
                assert m["seed"]==seed and m["heldout"]==writer
                assert m["n_training"]==78720 and m["n_heldout"]==640 and m["n_official_val"]==19200
                assert not m["independent_test_used"] and not m["professor_test_used"]
                assert not m["new_handmade_480_used"]
                assert m["writer_heldout_not_used_to_train"]
                assert m["handmade_train_preprocess"]=="CoM-only"
                assert m["heldout_and_official_val_preprocess"]=="CoM-only"
                assert m["frozen"]=={"hidden_layer_sizes":[256],"alpha":.0001,
                    "learning_rate_init":.001,"solver":"adam","activation":"relu",
                    "early_stopping":False,"max_iter":200,"tol":.0001,"n_iter_no_change":10}
                assert abs(m["augmentation"]["actual_ratio"]-.3)<.002
                preds=csvread(d/"oof_predictions.csv")
                assert len(preds)==640
                assert all(int(r["seed"])==seed and int(r["writer"])==writer and int(r["batch_size"])==batch for r in preds)
                flip=csvread(d/"class_family_flip.csv")
                assert len(flip)==64
                for row in flip:
                    assert row["dataset"] in ("OOF","OfficialVal") and float(row["scale"]) in (.8,1.2)
                    assert 0<=int(row["flip_count"])<=int(row["n"])
                assert all(m["focus_class"][cl]["n"]==40 for cl in "0123456789ABCDEF")
                artifacts[(batch,seed,writer)]={"m":m,"pred":preds,"flip":flip}
    assert len(artifacts)==36
    ref=artifacts[(200,42,41)]["m"]
    assert all(d["m"]["official_sha256"]==ref["official_sha256"] and
      d["m"]["handmade_sha256"]==ref["handmade_sha256"]
      for d in artifacts.values())
    metrics={bs:score_for(bs,artifacts) for bs in BATCHES}
    baseline=metrics[200]
    guard={bs:gates(baseline,metrics[bs]) for bs in BATCHES}
    assert all(guard[200].values())
    safe=[bs for bs in BATCHES if all(guard[bs].values())]
    strict=[bs for bs in safe if target(metrics[bs])]
    pool=strict or safe
    top=max(metrics[bs]["oof"] for bs in pool)
    near=[bs for bs in pool if metrics[bs]["oof"]>=top-.002-1e-12]
    # Existing CMB008 rule: within 0.2 pp, prioritize D->0,
    # then the other 3 confusions, then OOF, seed robustness and Val.
    near.sort(key=lambda bs:(metrics[bs]["pairs"]["D_to_0"],
      sum(metrics[bs]["pairs"][k] for k in PAIRS if k!="D_to_0"),
      -metrics[bs]["oof"],metrics[bs]["seed_std"],
      -metrics[bs]["official_val"],-bs))
    winner=near[0]
    same_oof=abs(baseline["oof"]-orig["oof"])<1e-12
    same_dirs=all(baseline["pairs"][p]==observed_dirs[p] for p in PAIRS)
    original_audit={"original_S5_es_false_oof":orig["oof"],
       "new_rerun_batch200_oof":baseline["oof"],
       "exact_oof_match":same_oof,
       "original_S5_four_confusions":observed_dirs,
       "rerun_batch200_four_confusions":baseline["pairs"],
       "exact_four_confusions_match":same_dirs}
    a.out.mkdir(parents=True,exist_ok=True)
    summary={"experiment":"EXP-CMB-008","stage":"S6_Batch_Size",
       "batch_sizes":list(BATCHES),"n_models":36,"n_seeds":3,"n_writers":4,
       "selected_candidate_research_only":winner,
       "selected_is_not_deployed":True,
       "recommended_condition_requires_user_approval":True,
       "strict_below_5pct_candidates":strict,
       "nonregression_candidates":safe,
       "selection_near_0p2pp":near,
       "same_dataset_SHA_across_all_36":True,
       "official_test_used":False,"professor_test_used":False,
       "new_handmade_480_used":False,"original_baseline_reproduction_audit":original_audit,
       "metrics":{str(bs):metrics[bs] for bs in BATCHES},
       "gates":{str(bs):guard[bs] for bs in BATCHES}}
    (a.out/"S6_results.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    rows=[]
    for bs in BATCHES:
        m=metrics[bs]
        rows.append({"batch_size":bs,"oof_accuracy":m["oof"],
          "official_val_accuracy":m["official_val"],
          "seed_std":m["seed_std"],"mean_epochs":m["mean_epochs"],
          "mean_fit_seconds":m["mean_fit_seconds"],
          "D_to_0":m["pairs"]["D_to_0"],
          "B_to_8":m["pairs"]["B_to_8"],
          "8_to_B":m["pairs"]["8_to_B"],
          "0_to_D":m["pairs"]["0_to_D"],
          "class_0_error":m["classerror"]["0"],
          "class_8_error":m["classerror"]["8"],
          "class_B_error":m["classerror"]["B"],
          "class_D_error":m["classerror"]["D"],
          "oof_max_flip":m["flip"]["OOF"],
          "official_val_max_flip":m["flip"]["OfficialVal"],
          "passes_guards":all(guard[bs].values()),
          "failed_guards":";".join(k for k,v in guard[bs].items() if not v),
          "target_all_four_below_5pct":target(m),
          "warnings":m["warnings"]})
    csvwrite(a.out/"S6_batch_size_comparison.csv",rows)
    paired={str(bs):compare_predictions(
      {(s,w):artifacts[(200,s,w)] for s in SEEDS for w in WRITERS},
      {(s,w):artifacts[(bs,s,w)] for s in SEEDS for w in WRITERS}) for bs in (128,64)}
    (a.out/"S6_paired_corrections.json").write_text(json.dumps(paired,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    notes=["# EXP-CMB-008 · S6 Batch Size 200/128/64 실측 비교",
      "",
      "> **결론은 개발용 추천이며 최종 HP 적용/모델 배포는 하지 않음.** Early Stopping=False는 유지. 3 Seed × 4 held-out Writer × 3 Batch Size = **36개 새 학습**.",
      "",
      "## 목표 및 고정 조건",
      "",
      "- 강의자료 09B 13번 슬라이드의 작은 Batch Size 비교 제안 검증. 바뀌는 항목은 **batch_size** 뿐이다.",
      "- 전처리 Size30% / Stroke28% / CoM, MLP (256,), ReLU, Adam, alpha=1e-4, lr_init=.001, early_stopping=False, max_iter=200. Official Train 76,800 + 다른 Writer 1,920장으로 학습, 보류 Writer 640장과 OfficialVal 19,200장 평가.",
      "- Original Test, 교수님 Test, 신규 480장 사용하지 않음. 원본 SHA·평가 파이프라인 동일.",
      "",
      "## 정량 결과",
      "",
      "| Batch Size | Writer OOF | OfficialVal | D→0 | B→8 | 8→B | 0→D | 가드레일 |",
      "|---:|---:|---:|---:|---:|---:|---:|---|"]
    for bs in BATCHES:
        m=metrics[bs]
        errors=m["pairs"]
        notes.append(f"| {bs} | {acc_str(m['oof'])} | {acc_str(m['official_val'])} | {errors['D_to_0']} | {errors['B_to_8']} | {errors['8_to_B']} | {errors['0_to_D']} | "+("통과" if all(guard[bs].values()) else "실패: "+", ".join(k for k,v in guard[bs].items() if not v))+" |")
    notes+=["","### Writer별 보류 OOF 및 학습 비용",
       "","| Batch | W41 | W42 | W43 | W44 | 평균 학습 Epoch | 평균 Fit Time (초) | 수렴 경고 |",
       "|---:|---:|---:|---:|---:|---:|---:|---:|"]
    for bs in BATCHES:
        m=metrics[bs]
        notes.append("| "+str(bs)+" | "+" | ".join(acc_str(m["writers"][str(w)]) for w in WRITERS)+
             f" | {m['mean_epochs']:.1f} | {m['mean_fit_seconds']:.1f} | {m['warnings']} |")
    notes+=["","### 주요 클래스별 전체 오분류율",
       "","| Batch | 0 | 8 | B | D | 최악 Writer OOF | 클래스별 최대 OOF Flip |",
       "|---:|---:|---:|---:|---:|---:|---:|"]
    for bs in BATCHES:
        m=metrics[bs]
        notes.append(f"| {bs} | "+" | ".join(acc_str(m["classerror"][cl]) for cl in FOCUS)+
          " | "+acc_str(min(m["writers"].values()))+" | "+acc_str(m["flip"]["OOF"])+" |")
    notes+=["","## 선정 규칙과 판단",
      "",
      "- **1순위:** D→0 / B→8 / 8→B / 0→D 4방향 각각 <5% (480개에서 23건 이하), 동시에 모든 가드레일 통과.",
      "- **미달 시:** Writer OOF가 기준선보다 낮지 않고, D→0 등 네 방향과 class 0/8/B/D 오류가 증가하지 않으며, Writer별 OOF 최소 1%p/OfficialVal 최대 0.5%p 손실 제한 및 최대 클래스 가족 Flip 증가 5%p 이내의 후보만 선정.",
      "- **가드레일 통과 후보 중** OOF 최고 정확도와 0.2%p 이내인 후보는 D→0을 우선하고 나머지 세 방향 합계로 순서를 매긴다.",
      f"- **검증 결과:** 4방향 엄격 목표 통과 후보: {strict or '없음'}. 안전 가드 통과 후보: {safe}. **개발용 추천 Batch Size: {winner}**.",
      "- 후보의 OOF가 높더라도 한 방향·취약 클래스가 악화되면 전체 정확도만으로 변경하지 않는다.",
      "",
      "## 기준선 재실행 감사",
      "",
      f"- 원본 S5 ES=False OOF: **{acc_str(orig['oof'])}**. 이번 새 Batch=200 OOF: **{acc_str(baseline['oof'])}**. 원본과 전체 정확도 일치 여부: **{same_oof}**.",
      f"- 4방향 오류 완전 일치 여부: **{same_dirs}**. 기존 연구의 분할·환경·실행 차이에 따른 재현성 차이가 나타나면 수치를 숨기지 않는다.",
      "- **이번 S6 후보 비교·선정은 새 36개 실행에 포함된 동일 Batch=200 기준선에 상대 비교했다.** 원본 S5와 수치가 다르더라도 이를 원본 S5를 덮어쓴 결과로 취급하지 않는다.",
      "",
      "## 동일 7,680건 예측에서 교정과 신규 오류",
      "",
      "| 후보 vs Batch=200 | 기존 오답 교정 | 기존 정답이 오답으로 변함 | 순개선 |",
      "|---:|---:|---:|---:|"]
    for bs in (128,64):
        q=paired[str(bs)]
        notes.append(f"| {bs} | {q['fixed']} | {q['induced']} | {q['net_gain']:+d} |")
    notes+=["","## 후속 결정",
      "",
      f"- **추천 조건: Batch Size {winner}** (개발용 분석 결과). **현재 기준선 200의 HP와 모델을 자동으로 바꾸지 않는다.**",
      "- 기존 Early Stopping=False 확정 상태와 무관하게, 이번 실험은 batch_size만 변경했다.",
      "- 동일 보류 Writer 4명을 반복 활용한 개발용 OOF 선택 편향 가능성은 남아 있으며, 독립 Test나 최종 배포 전에 별도 승인 및 검증이 필요하다.",
      "",
      "## 원본 추적",
      "",
      "- [정량 CSV](../../results/Hyperparameter_Optimization/S6_Batch_Size/S6_batch_size_comparison.csv)",
      "- [종합·가드레일 JSON](../../results/Hyperparameter_Optimization/S6_Batch_Size/S6_results.json)",
      "- [오답 교정/유발 JSON](../../results/Hyperparameter_Optimization/S6_Batch_Size/S6_paired_corrections.json)",
      "- [S6 학습 코드](../../scripts/exp_cmb008_s6_batch_size.py)",
      "- [S6 집계 코드](../../scripts/exp_cmb008_s6_batch_aggregate.py)",
      ""]
    (a.out/"S6_Results.md").write_text("\n".join(notes),encoding="utf-8")
    print("BATCH_COMPARE_DONE",json.dumps({
       "recommended_batch":winner,
       "batch200_reproduced_original_oof":same_oof,
       "batch200_reproduced_original_four_dirs":same_dirs,
       "metrics":{str(k):{"oof":v["oof"],"official":v["official_val"],"pairs":v["pairs"],
          "guards":guard[k]} for k,v in metrics.items()}},ensure_ascii=False),flush=True)

if __name__=="__main__":
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")